In [69]:
import pandas as pd
import numpy as np
import sys
import os
import joblib
import xgboost as xgb

In [70]:
from pathlib import Path
# Find the project root (the folder that contains src/config.py) and make src/ importable
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import MOLECULES_UNIQUE, MATRIX_IDS, TRAIN, TEST, MODEL_HANSEN, ensure_dir

from controller.predictor_runner import generate_ml_ready_file_hansen

va chercher le fichier ml ready file onehot dnas predictor runner

ici on va transformer nos smiles en fingerprint de type Morgan. Avant ça elle le transforme en objet rdkit. puis a la fin conversion du format rdkit en format tableau numpy Le Type Morgan est une circular fingerprint. 

In [71]:
solvent_dict = {
    'MeOH': {'Hansen_D': 15.1, 'Hansen_P': 12.3, 'Hansen_H': 22.3},
    'ACN': {'Hansen_D': 15.3, 'Hansen_P': 18.0, 'Hansen_H': 6.1},
    'DMSO': {'Hansen_D': 18.4, 'Hansen_P': 16.4, 'Hansen_H': 10.2},
    'DCM': {'Hansen_D': 18.2, 'Hansen_P': 6.3, 'Hansen_H': 6.1},
    'CHCl3': {'Hansen_D': 17.8, 'Hansen_P': 3.1, 'Hansen_H': 5.7}
}

print(f"Solvants configurés : {list(solvent_dict.keys())}")
for name, params in solvent_dict.items():
    print(f"  {name} : {params}")

Solvants configurés : ['MeOH', 'ACN', 'DMSO', 'DCM', 'CHCl3']
  MeOH : {'Hansen_D': 15.1, 'Hansen_P': 12.3, 'Hansen_H': 22.3}
  ACN : {'Hansen_D': 15.3, 'Hansen_P': 18.0, 'Hansen_H': 6.1}
  DMSO : {'Hansen_D': 18.4, 'Hansen_P': 16.4, 'Hansen_H': 10.2}
  DCM : {'Hansen_D': 18.2, 'Hansen_P': 6.3, 'Hansen_H': 6.1}
  CHCl3 : {'Hansen_D': 17.8, 'Hansen_P': 3.1, 'Hansen_H': 5.7}


ici on fait un dictionnaire avec pour chaque solvent son smile associé. Il faut faire ça car dans notre document de test on doit pouvoir associé le nom de la colonne avec smiles qui est associé en fingerprint 

In [72]:
matrix_ids_path = MATRIX_IDS
all_matrix_ids = joblib.load(matrix_ids_path)

print(f"Matrix IDs chargées : {len(all_matrix_ids)}")

Matrix IDs chargées : 1


In [73]:
train_path = ensure_dir(TRAIN)
df_train = pd.read_csv(train_path)
df_train.columns = df_train.columns.str.strip()

print(f"Fichier chargé : {train_path}")
print(f"Nombre de lignes : {len(df_train)}")
print(f"Colonnes : {df_train.columns.tolist()}")

Fichier chargé : /home/coralie/workspace/ProjectII/solubility_prediction_nmr/data/train.csv
Nombre de lignes : 122
Colonnes : ['Column', 'Method', 'Sample Name', 'CAS', 'RT', 'Signal', 'Matrix ID Name', 'MeOH', 'ACN', 'DMSO', 'DCM', 'CHCl3', 'solute_smiles']


Ici on va chercher le file de train_path pour faire un nouveau dossier train avec les valeurs. est ce que ducoup on va directement modifier le train_path ? 

In [74]:
def build_all_solvents_df(cleaned_df, solvent_dict, all_matrix_ids):
    all_dfs = []
    for solvent_name in solvent_dict.keys():
        if solvent_name not in cleaned_df.columns:
            print(f"Colonne '{solvent_name}' absente, ignorée.")
            continue
        print(f"Traitement du solvant : {solvent_name}...")
        solvent_ml_df = generate_ml_ready_file_hansen(
            cleaned_df, solvent_name, solvent_dict, all_matrix_ids
        )
        solvent_ml_df['Solvent_Name'] = solvent_name   # ← nouvelle ligne
        print(f"  → {len(solvent_ml_df)} lignes générées pour {solvent_name}")
        all_dfs.append(solvent_ml_df)
    return pd.concat(all_dfs, ignore_index=True)

on veut entrainer tous les solvants en même temps donc on créer un format empilé. Ensuite la fonction filtre les lignes valides pour ce solvant, calcule les fingerprints, et prépare le one-hot. chaque résultat pour un solvent est stocké dans une liste. 

In [75]:
print("Construction du dataset d'entraînement (tous solvants empilés)...")
train_ml_df = build_all_solvents_df(df_train, solvent_dict, all_matrix_ids)

print(f"\nTerminé : {len(train_ml_df)} lignes au total (molécules × solvants)")
train_ml_df.head()

Construction du dataset d'entraînement (tous solvants empilés)...
Traitement du solvant : MeOH...
  → 122 lignes générées pour MeOH
Traitement du solvant : ACN...
  → 122 lignes générées pour ACN
Traitement du solvant : DMSO...
  → 122 lignes générées pour DMSO
Traitement du solvant : DCM...
  → 122 lignes générées pour DCM
Traitement du solvant : CHCl3...
  → 122 lignes générées pour CHCl3

Terminé : 610 lignes au total (molécules × solvants)


,MolWt,RT,Matrix_Onehot,Solvent_Hansen,Soluble,SMILES,Solvent_Name
0,185.064,5.607,[1],"[15.1, 12.3, 22.3]",1,CC(C1=CC=CC=C1)Br,MeOH
1,266.253,4.021,[1],"[15.1, 12.3, 22.3]",0,C1=CC=C(C=C1)COC(=O)NCC(=O)NCC(=O)O,MeOH
2,341.363,3.327,[1],"[15.1, 12.3, 22.3]",0,C1=CC=C2C(=C1)C(C3=CC=CC=C32)COC(=O)N[C@@H](CC...,MeOH
3,272.284,5.678,[1],"[15.1, 12.3, 22.3]",0,C1C[C@H](P(=O)([C@@H]1C2=CC=CC=C2)O)C3=CC=CC=C3,MeOH
4,473.569,6.019,[1],"[15.1, 12.3, 22.3]",0,CC(C)(C)OC1=CC=C(C=C1)CC[C@@H](C(=O)O)NC(=O)OC...,MeOH


on assure que le calcul démarre. print final donne total de ligne obtenu pour 85% des mol et chaque fois 5 solvent. puis on montre les 5 première lignes du résultats.

In [76]:
X_train_num = train_ml_df[['MolWt', 'RT']].values
X_train_matrix = np.vstack(train_ml_df['Matrix_Onehot'].values)
X_train_hansen = np.vstack(train_ml_df['Solvent_Hansen'].values)
X_train = np.hstack((X_train_num, X_train_matrix, X_train_hansen))
y_train = train_ml_df['Soluble'].values

print(f"X_train shape : {X_train.shape}")
print(f"y_train shape : {y_train.shape}")

X_train shape : (610, 6)
y_train shape : (610,)


ici chaque cellule de trainmldf contient un tableau de 1024 valeurs. ici on va emplier tous ces tableaux pour former une matrice 2D. Ici ça va coller les matrices pour les solvent et pour les molécules à côté. A la fin il associe soluble ou insoluble. ça je comprends pas !

In [77]:
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
scale_weight = neg_count / pos_count if pos_count > 0 else 1.0

print(f"Exemples négatifs (insoluble) : {neg_count}")
print(f"Exemples positifs (soluble)   : {pos_count}")
print(f"scale_pos_weight calculé      : {scale_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_weight,
    n_jobs=-1
)

print("\nEntraînement en cours...")
model.fit(X_train, y_train)
print("Entraînement terminé !")

Exemples négatifs (insoluble) : 203
Exemples positifs (soluble)   : 407
scale_pos_weight calculé      : 0.50

Entraînement en cours...
Entraînement terminé !


il compte ici combien d'exemple sont soluble et insoluble. on regarde que il surestime pas le DMSO avec rundescriptorcompetition. Ensuite c'est le modèle: nbr d'arbres de décision, vitesse d'apprentissage, profondeur max, compenser déséquilibre, utilisation des coeurs CPU pour accélérer l'entrainement. 

In [78]:
import joblib

model_path = ensure_dir(MODEL_HANSEN)
joblib.dump(model, model_path)

print(f"Modèle sauvegardé : {model_path}")

Modèle sauvegardé : /home/coralie/workspace/ProjectII/solubility_prediction_nmr/data/xgboost_all_solvents_model.joblib


ici on sauvegarde le fichier crée pour pouvoir l'utiliser dans test après 